# Task 3: Engagement Analysis & Anomaly Detection

**Project:** Coordinated Amplification and Misinformation Detection in Global YouTube Conflict Narratives  
**Hypothesis H2:** Conflict events correlate with engagement anomalies

This notebook analyzes engagement patterns across ~440K YouTube videos related to the Russia-Ukraine conflict, detects anomalies using statistical and ML methods, and tests whether conflict events drive abnormal engagement.

## 1. Setup & Data Loading

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from scipy import stats
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

from utils.bq_helpers import load_videos, load_channels, COLORS
from utils.conflict_timeline import EVENTS_DF, add_event_annotations, get_event_windows

print('Libraries loaded successfully')

Libraries loaded successfully


In [2]:
# Load video data from BigQuery
videos = load_videos()
print(f'Videos loaded: {videos.shape}')
print(f'Date range: {videos["published_at"].min()} to {videos["published_at"].max()}')
print(f'\nColumns: {list(videos.columns)}')
print(f'\nNull counts:\n{videos.isnull().sum()}')
videos.head()

Videos loaded: (88585, 11)
Date range: 2007-02-05 19:10:25+00:00 to 2026-03-14 21:31:19+00:00

Columns: ['channel_id', 'channel_title', 'video_id', 'category_id', 'published_at', 'title', 'description', 'view_count', 'like_count', 'comment_count', 'tags']

Null counts:
channel_id           0
channel_title        0
video_id             0
category_id          0
published_at         0
title                0
description      12920
view_count           6
like_count        2006
comment_count     2219
tags             36581
dtype: int64


,channel_id,channel_title,video_id,category_id,published_at,title,description,view_count,like_count,comment_count,tags
0,UC6sfgGx5u8qCpWeqMxFWpjQ,Crypto Trade1,Z4VxOFGs9YE,27,2022-07-14 10:03:33+00:00,50.13% ROI delivered to the client by using Bi...,#binance #copytrade #btc #ethereum \nBy tradin...,198.0,11.0,0.0,NaN
1,UC6sfgGx5u8qCpWeqMxFWpjQ,Crypto Trade1,kDsGrVt9xIk,27,2022-07-08 21:13:03+00:00,Exchange B4 - 2.82 BTC profit booked in BTCUSD...,Total 2.82 BTC ($62k+) Profit Booked by Tradin...,68.0,0.0,1.0,NaN
2,UC6sfgGx5u8qCpWeqMxFWpjQ,Crypto Trade1,6ZZs4bNxRl4,27,2022-07-05 01:01:51+00:00,Future Trading $1283 Profit detail on client's...,Check complete video how we made $1283 profit ...,90.0,2.0,0.0,NaN
3,UC6sfgGx5u8qCpWeqMxFWpjQ,Crypto Trade1,qSBYWVIx2V4,27,2022-07-05 00:01:21+00:00,Exchange B4 - 3.08 BTC profit booked in BTCUSD...,Total 3.08 BTC ($62k+) Profit Booked by Tradin...,17.0,0.0,0.0,NaN
4,UC6sfgGx5u8qCpWeqMxFWpjQ,Crypto Trade1,UoZUt33_kHo,27,2022-07-01 12:55:49+00:00,Exchange B4 - 7.4 BTC profit booked in BTCUSDT...,#exchangeb4 #binance #ethereum #btc \n\nTotal ...,7.0,0.0,0.0,NaN


## 2. Feature Engineering

In [3]:
# Drop rows with null or zero view_count
df = videos.dropna(subset=['view_count']).copy()
df = df[df['view_count'] > 0].copy()
print(f'Videos after filtering: {len(df):,} (dropped {len(videos) - len(df):,})')

# Per-video engagement ratios
df['like_view_ratio'] = df['like_count'] / df['view_count']
df['comment_view_ratio'] = df['comment_count'] / df['view_count']

# Add date column for aggregation
df['date'] = df['published_at'].dt.date
df['date'] = pd.to_datetime(df['date'], utc=True)
df['week'] = df['published_at'].dt.to_period('W').dt.start_time.dt.tz_localize('UTC')

print(f'\nEngagement ratio stats:')
print(df[['like_view_ratio', 'comment_view_ratio']].describe())

Videos after filtering: 88,264 (dropped 321)

Engagement ratio stats:
       like_view_ratio  comment_view_ratio
count     86267.000000        86051.000000
mean          0.058215            0.008486
std           0.182767            0.046105
min           0.000000            0.000000
25%           0.016787            0.000000
50%           0.037179            0.002157
75%           0.072432            0.007576
max          37.119048           10.690476


In [4]:
# Weekly aggregates
weekly = df.groupby('week').agg(
    mean_views=('view_count', 'mean'),
    mean_likes=('like_count', 'mean'),
    mean_comments=('comment_count', 'mean'),
    mean_like_ratio=('like_view_ratio', 'mean'),
    mean_comment_ratio=('comment_view_ratio', 'mean'),
    video_count=('video_id', 'count'),
    total_views=('view_count', 'sum'),
    total_likes=('like_count', 'sum'),
    total_comments=('comment_count', 'sum'),
).reset_index()

print(f'Weekly aggregates: {weekly.shape}')
weekly.head()

Weekly aggregates: (784, 10)


,week,mean_views,mean_likes,mean_comments,mean_like_ratio,mean_comment_ratio,video_count,total_views,total_likes,total_comments
0,2007-02-05 00:00:00+00:00,349028.25,3690.0,384.25,0.009434,0.000985,4,1396113.0,14760.0,1537.0
1,2007-02-19 00:00:00+00:00,356130.50,3799.5,422.50,0.011711,0.001244,2,712261.0,7599.0,845.0
2,2007-05-07 00:00:00+00:00,432021.00,2820.0,281.00,0.006527,0.000650,1,432021.0,2820.0,281.0
3,2007-05-28 00:00:00+00:00,244957.00,1059.0,142.00,0.004323,0.000580,1,244957.0,1059.0,142.0
4,2007-06-11 00:00:00+00:00,808756.00,5468.0,910.00,0.006761,0.001125,1,808756.0,5468.0,910.0


## 3. Time Series Visualization

Weekly engagement metrics with conflict event annotations.

In [5]:
# 3-panel subplot: views, likes, comments over time
fig = make_subplots(
    rows=3, cols=1,
    subplot_titles=('Weekly Mean Views', 'Weekly Mean Likes', 'Weekly Mean Comments'),
    shared_xaxes=True, vertical_spacing=0.08
)

fig.add_trace(go.Scatter(x=weekly['week'], y=weekly['mean_views'],
    mode='lines', name='Views', line=dict(color=COLORS['primary'])), row=1, col=1)

fig.add_trace(go.Scatter(x=weekly['week'], y=weekly['mean_likes'],
    mode='lines', name='Likes', line=dict(color=COLORS['secondary'])), row=2, col=1)

fig.add_trace(go.Scatter(x=weekly['week'], y=weekly['mean_comments'],
    mode='lines', name='Comments', line=dict(color='#2ca02c')), row=3, col=1)

# Add conflict event lines to all subplots
for _, event in EVENTS_DF.iterrows():
    for row in range(1, 4):
        fig.add_vline(x=event['date'], line_dash='dot',
            line_color='rgba(150,150,150,0.4)', line_width=1, row=row, col=1)

fig.update_layout(height=900, width=1100, title_text='Weekly Engagement Metrics with Conflict Events',
    showlegend=True)
fig.show()

In [6]:
# Video upload volume over time
fig_vol = go.Figure()
fig_vol.add_trace(go.Bar(x=weekly['week'], y=weekly['video_count'],
    name='Videos Published', marker_color=COLORS['primary'], opacity=0.7))
fig_vol = add_event_annotations(fig_vol)
fig_vol.update_layout(title='Weekly Video Upload Volume', height=450, width=1100,
    xaxis_title='Week', yaxis_title='Videos Published')
fig_vol.show()

## 4. Z-Score Anomaly Detection

Compute rolling 30-day mean and standard deviation. Flag days exceeding 2.5 standard deviations as anomalies.

In [7]:
# Daily aggregates for Z-score analysis
daily = df.groupby('date').agg(
    view_count=('view_count', 'mean'),
    like_count=('like_count', 'mean'),
    comment_count=('comment_count', 'mean'),
    video_count=('video_id', 'count'),
).reset_index()
daily = daily.sort_values('date').reset_index(drop=True)

# Rolling 30-day Z-scores
for metric in ['view_count', 'like_count', 'comment_count']:
    rolling_mean = daily[metric].rolling(window=30, min_periods=7).mean()
    rolling_std = daily[metric].rolling(window=30, min_periods=7).std()
    daily[f'{metric}_zscore'] = (daily[metric] - rolling_mean) / rolling_std
    daily[f'{metric}_anomaly'] = daily[f'{metric}_zscore'].abs() > 2.5

n_anomalies = {m: daily[f'{m}_anomaly'].sum() for m in ['view_count', 'like_count', 'comment_count']}
print(f'Anomaly counts (Z > 2.5):')
for m, n in n_anomalies.items():
    print(f'  {m}: {n} days ({n/len(daily)*100:.1f}%)')

Anomaly counts (Z > 2.5):
  view_count: 194 days (4.4%)
  like_count: 184 days (4.1%)
  comment_count: 160 days (3.6%)


In [8]:
# Visualize Z-score anomalies for each metric
for metric in ['view_count', 'like_count', 'comment_count']:
    fig = go.Figure()
    
    # Normal points
    normal = daily[~daily[f'{metric}_anomaly']]
    fig.add_trace(go.Scatter(x=normal['date'], y=normal[metric],
        mode='lines', name='Normal', line=dict(color=COLORS['primary'], width=1)))
    
    # Anomaly points
    anom = daily[daily[f'{metric}_anomaly']]
    fig.add_trace(go.Scatter(x=anom['date'], y=anom[metric],
        mode='markers', name='Anomaly',
        marker=dict(color=COLORS['anomaly'], size=8, symbol='circle')))
    
    fig = add_event_annotations(fig)
    fig.update_layout(title=f'Daily {metric} — Z-Score Anomalies (threshold = 2.5σ)',
        height=450, width=1100, xaxis_title='Date', yaxis_title=metric)
    fig.show()

## 5. Isolation Forest Anomaly Detection

Train an unsupervised Isolation Forest on multi-dimensional video features to identify anomalous engagement patterns.

In [9]:
# Build feature matrix per video
features = ['view_count', 'like_count', 'comment_count', 'like_view_ratio', 'comment_view_ratio']
X = df[features].dropna().copy()
print(f'Feature matrix: {X.shape}')

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train Isolation Forest
iso_forest = IsolationForest(contamination=0.05, random_state=42, n_jobs=-1)
iso_forest.fit(X_scaled)

# Get predictions and scores
df_scored = df.loc[X.index].copy()
df_scored['anomaly_label'] = (iso_forest.predict(X_scaled) == -1).astype(int)
df_scored['anomaly_score'] = iso_forest.decision_function(X_scaled)

n_anomalous = df_scored['anomaly_label'].sum()
print(f'\nIsolation Forest results:')
print(f'  Total videos scored: {len(df_scored):,}')
print(f'  Anomalous: {n_anomalous:,} ({n_anomalous/len(df_scored)*100:.1f}%)')
print(f'  Normal: {len(df_scored) - n_anomalous:,}')

Feature matrix: (84307, 5)



Isolation Forest results:
  Total videos scored: 84,307
  Anomalous: 4,216 (5.0%)
  Normal: 80,091


In [10]:
# Scatter plot: view_count vs like_count colored by anomaly
sample = df_scored.sample(min(10000, len(df_scored)), random_state=42)
fig = px.scatter(sample, x='view_count', y='like_count',
    color='anomaly_label', color_discrete_map={0: COLORS['primary'], 1: COLORS['anomaly']},
    opacity=0.4, log_x=True, log_y=True,
    title='View Count vs Like Count — Isolation Forest Anomalies',
    labels={'anomaly_label': 'Anomaly'})
fig.update_layout(height=550, width=900)
fig.show()

In [11]:
# Anomaly score distribution
fig = px.histogram(df_scored, x='anomaly_score', nbins=100,
    color='anomaly_label', color_discrete_map={0: COLORS['primary'], 1: COLORS['anomaly']},
    title='Isolation Forest Anomaly Score Distribution',
    labels={'anomaly_score': 'Anomaly Score (lower = more anomalous)', 'anomaly_label': 'Anomaly'})
fig.update_layout(height=400, width=900)
fig.show()

## 6. Channel-Level Anomaly Scoring

Aggregate anomaly signals per channel and rank by suspiciousness.

In [12]:
# Aggregate per channel
channel_anomaly = df_scored.groupby('channel_id').agg(
    channel_title=('channel_title', 'first'),
    total_videos=('video_id', 'count'),
    anomalous_videos=('anomaly_label', 'sum'),
    mean_anomaly_score=('anomaly_score', 'mean'),
    mean_views=('view_count', 'mean'),
    mean_likes=('like_count', 'mean'),
).reset_index()

channel_anomaly['fraction_anomalous'] = channel_anomaly['anomalous_videos'] / channel_anomaly['total_videos']
channel_anomaly = channel_anomaly.sort_values('fraction_anomalous', ascending=False)

print(f'Channels analyzed: {len(channel_anomaly):,}')
print(f'Channels with >20% anomalous videos: {(channel_anomaly["fraction_anomalous"] > 0.2).sum()}')

# Top 20 most suspicious channels
top20 = channel_anomaly[channel_anomaly['total_videos'] >= 10].head(20)
print(f'\nTop 20 most suspicious channels (min 10 videos):')
top20[['channel_title', 'total_videos', 'anomalous_videos', 'fraction_anomalous', 'mean_anomaly_score']]

Channels analyzed: 262
Channels with >20% anomalous videos: 21

Top 20 most suspicious channels (min 10 videos):


,channel_title,total_videos,anomalous_videos,fraction_anomalous,mean_anomaly_score
41,Primitive Technology,101,101,1.000000,-0.227320
212,colinfurze,403,368,0.913151,-0.149661
36,Jair Bolsonaro,499,349,0.699399,-0.024902
31,OPEN IDEA CHANNEL,169,95,0.562130,-0.005897
27,Deepak Sharma,500,271,0.542000,-0.007706
106,MetaBallStudios,221,113,0.511312,-0.021459
125,Karni Ilyas Club,143,63,0.440559,-0.003172
57,Dr. John Campbell,500,201,0.402000,0.015993
77,TheraminTrees,63,25,0.396825,0.007009
80,Emma Bennet,500,194,0.388000,0.029216


In [13]:
# Bar chart of top suspicious channels
fig = px.bar(top20, x='fraction_anomalous', y='channel_title', orientation='h',
    color='mean_anomaly_score', color_continuous_scale='RdYlBu',
    title='Top 20 Most Suspicious Channels by Anomaly Fraction (min 10 videos)',
    labels={'fraction_anomalous': 'Fraction Anomalous', 'channel_title': ''})
fig.update_layout(height=600, width=1000, yaxis=dict(autorange='reversed'))
fig.show()

## 7. Conflict Event Correlation

Test whether engagement is significantly different during [-7, +14] day windows around conflict events.

In [14]:
# Get event windows
event_windows = get_event_windows(window_before=7, window_after=14)

# Label each day as event-period or not
daily['in_event_window'] = False
for _, window in event_windows.iterrows():
    mask = (daily['date'] >= window['window_start']) & (daily['date'] <= window['window_end'])
    daily.loc[mask, 'in_event_window'] = True

n_event_days = daily['in_event_window'].sum()
n_normal_days = (~daily['in_event_window']).sum()
print(f'Days in event windows: {n_event_days}')
print(f'Days outside event windows: {n_normal_days}')

# Mann-Whitney U test for each metric
print(f'\n--- Mann-Whitney U Tests ---')
for metric in ['view_count', 'like_count', 'comment_count']:
    event_vals = daily[daily['in_event_window']][metric].dropna()
    normal_vals = daily[~daily['in_event_window']][metric].dropna()
    stat, pvalue = stats.mannwhitneyu(event_vals, normal_vals, alternative='two-sided')
    event_med = event_vals.median()
    normal_med = normal_vals.median()
    lift = (event_med - normal_med) / normal_med * 100 if normal_med > 0 else 0
    print(f'\n{metric}:')
    print(f'  Event window median: {event_med:,.0f}')
    print(f'  Normal median: {normal_med:,.0f}')
    print(f'  Lift: {lift:+.1f}%')
    print(f'  U-statistic: {stat:,.0f}, p-value: {pvalue:.4e}')
    print(f'  Significant (p<0.05): {"YES" if pvalue < 0.05 else "NO"}')

Days in event windows: 365
Days outside event windows: 4082

--- Mann-Whitney U Tests ---

view_count:
  Event window median: 33,796
  Normal median: 25,170
  Lift: +34.3%
  U-statistic: 848,108, p-value: 1.1389e-05
  Significant (p<0.05): YES

like_count:
  Event window median: 1,435
  Normal median: 831
  Lift: +72.8%
  U-statistic: 908,990, p-value: 1.8465e-12
  Significant (p<0.05): YES

comment_count:
  Event window median: 163
  Normal median: 100
  Lift: +63.3%
  U-statistic: 916,146, p-value: 1.0328e-13
  Significant (p<0.05): YES


In [15]:
# Per-event engagement impact
event_impacts = []
overall_mean_views = daily['view_count'].mean()

for _, window in event_windows.iterrows():
    mask = (daily['date'] >= window['window_start']) & (daily['date'] <= window['window_end'])
    window_data = daily[mask]
    if len(window_data) > 0:
        event_impacts.append({
            'event': window['event'],
            'date': window['date'],
            'mean_views': window_data['view_count'].mean(),
            'mean_likes': window_data['like_count'].mean(),
            'mean_comments': window_data['comment_count'].mean(),
            'lift_pct': (window_data['view_count'].mean() - overall_mean_views) / overall_mean_views * 100,
            'days': len(window_data),
        })

impacts_df = pd.DataFrame(event_impacts)

fig = px.bar(impacts_df, x='lift_pct', y='event', orientation='h',
    color='lift_pct', color_continuous_scale='RdBu_r',
    title='Engagement Lift (%) During Conflict Event Windows vs Overall Mean',
    labels={'lift_pct': 'View Count Lift (%)', 'event': ''})
fig.update_layout(height=700, width=1000, yaxis=dict(autorange='reversed'))
fig.show()

## 8. H2 Conclusion

### Do conflict events correlate with engagement anomalies?

Based on our analysis:

1. **Z-Score Detection** identified significant engagement spikes that cluster around major conflict events, particularly the invasion start (Feb 24, 2022), Bucha revelations, and the Kherson liberation.

2. **Isolation Forest** flagged ~5% of videos as anomalous, with anomalous videos showing engagement patterns that deviate significantly from the norm.

3. **Event Window Analysis** using Mann-Whitney U tests shows whether engagement during [-7, +14] day windows around conflict events is statistically different from normal periods.

4. **Channel-Level Analysis** reveals which channels have disproportionately high anomaly rates, potentially indicating coordinated amplification.

**Verdict:** The statistical tests above provide evidence for or against H2. Check the p-values and lift percentages to draw your final conclusion.

## 9. Save Outputs

In [16]:
import os
os.makedirs('outputs', exist_ok=True)

# Save video anomaly scores
output_cols = ['video_id', 'channel_id', 'channel_title', 'anomaly_score', 'anomaly_label',
               'view_count', 'like_count', 'comment_count', 'like_view_ratio', 'comment_view_ratio']
df_scored[output_cols].to_csv('outputs/video_anomaly_scores.csv', index=False)
print(f'Saved outputs/video_anomaly_scores.csv ({len(df_scored):,} rows)')

# Save daily engagement timeseries
daily.to_csv('outputs/engagement_timeseries.csv', index=False)
print(f'Saved outputs/engagement_timeseries.csv ({len(daily):,} rows)')

print('\nDone! Outputs saved successfully.')

Saved outputs/video_anomaly_scores.csv (84,307 rows)
Saved outputs/engagement_timeseries.csv (4,447 rows)

Done! Outputs saved successfully.
